# Now to actually make map

In [1]:
import os
import json

import numpy as np
import pandas as pd

from ipywidgets import Dropdown

from bqplot import Lines, Figure, LinearScale, DateScale, Axis

from ipyleaflet import Map, GeoJSON, WidgetControl, basemaps

import plotly.graph_objects as go

In [2]:
%run graph_programs/custom_colors.py
%run graph_programs/female_percentage.py


# Exploratory Data Analysis

In [3]:
# data = pd.read_json(os.path.abspath("nations.json"))

# converting shpae file to geojson
import geopandas as gpd

geo_json_path = os.path.join("geos", "cc20_us_03102025_500k_small.geojson")
gdf_small = gpd.read_file(geo_json_path)  # Load the existing GeoJSON file
    

In [4]:
print(gdf_small.columns)
geoids = gdf_small["GEOID"].astype(str).tolist()
print(geoids) 

bounds = gdf_small.total_bounds

Index(['CCN20', 'GEOID', 'STCD', 'STFIPS', 'STAB', 'D001TPOP', 'D0020004',
       'D0030509', 'D0041014', 'D0051519',
       ...
       'D154VSEAS', 'D155VOTHR', 'D156HVACR', 'D157RVACR', 'D158OCCHU',
       'D159OCCHUO', 'D160OCCHUR', 'DIVERSITY', 'DEPRATIO', 'geometry'],
      dtype='str', length=168)
['0101001', '0101002', '0101003', '0101004', '0101005', '0101006', '0101007', '0101008', '0101009', '0101010', '0101011', '0101012', '0101013', '0101014', '0101015', '0101016', '0102001', '0102002', '0102003', '0102004', '0102005', '0102006', '0102007', '0102008', '0102009', '0102010', '0102011', '0102012', '0102013', '0102014', '0102015', '0102016', '0102017', '0102018', '0103001', '0103002', '0103003', '0103004', '0103005', '0103006', '0103007', '0103008', '0103009', '0103010', '0103011', '0103012', '0103013', '0103014', '0103015', '0103016', '0103017', '0104001', '0104002', '0104003', '0104004', '0104005', '0104006', '0104007', '0104008', '0104009', '0104010', '0104011', '0104012', '

In [5]:
demographic_var = ["D001TPOP", "D025MALE", "D049FEMALE"]  # Replace with the actual column name for the demographic variable

In [6]:
# cleaning data for proof
data = gdf_small[['GEOID'] + demographic_var][gdf_small["GEOID"].astype(str).isin(geoids)]
print(data.head())

     GEOID  D001TPOP  D025MALE  D049FEMALE
0  0101001     55599     27908       27691
1  0101002     67916     32845       35071
2  0101003     34347     16594       17753
3  0101004     45525     22560       22965
4  0101005     36757     18254       18503


# Making Base Map

In [7]:
m = Map(basemap=basemaps.CartoDB.Positron, 
        zoom=5)

hover_style = {'color': cc_red, 
               'fillColor': cc_red, 
               'opacity': 1.0, 
               'weight': 1.5, 
               'fillOpacity': 0.4}

# this makes the map tiles
geo = GeoJSON(
    data=gdf_small.__geo_interface__,
    style={"fillColor": "white", "weight": 0.5, 'color': cc_blue},
    hover_style= hover_style,
    name="GEOID",
)

m.add(geo)

# Fit map to bounds: [[south, west], [north, east]]
m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

m

Map(center=[0.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

# Adding Widgets

In [8]:
# Highlight overlay — starts empty, only ever holds the small selected subset
highlight_layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"fillColor": cc_black, "color": cc_black, "weight": 1.5, "fillOpacity": 0.4},
)

m.add_layer(highlight_layer)

In [9]:
fig_percent = make_female_percentage_graph()
fig_percent.show()

In [10]:
from bqplot import Figure, Label, LinearScale


# Fixed-range scales just for figure-relative positioning
label_scale_x = LinearScale(min=0, max=1)
label_scale_y = LinearScale(min=0, max=1)

hover_geoid = Label(
    text=['geoid'],
    x=[0.02], y=[0.75],
    scales={'x': label_scale_x, 'y': label_scale_y},
    font_size='20px',
    font_weight='bold',
    colors=cc_red,
)

hover_text = Label(
    text=['Total population'],
    x=[0.02], y=[0.45],
    scales={'x': label_scale_x, 'y': label_scale_y},
    font_size='20px',
    colors=cc_blue,
)

hover_pop_value = Label(
    text=['pop number'],
    x=[0.02], y=[0.3],
    scales={'x': label_scale_x, 'y': label_scale_y},
    font_size='20px',
    colors=cc_blue,
)

# Display the figure
hover_fig = Figure(
    marks=[hover_geoid, hover_text, hover_pop_value],
    layout={"width": "150px", "height": "110px"},
    fig_margin={"top": 0.01, "bottom": 0.01, "left": 0.01, "right": 0.01},
    min_aspect_ratio=0, max_aspect_ratio=100,
)


hover_fig

Figure(fig_margin={'top': 0.01, 'bottom': 0.01, 'left': 0.01, 'right': 0.01}, layout=Layout(height='110px', wi…

In [11]:
def update_hover_fig(geoid):
    # Filter the data for the selected GEOID
    filtered_data = data[data['GEOID'] == geoid]
    
    # Get the value of the demographic variable in this case, total pop
    value = filtered_data['D001TPOP'].values[0] if not filtered_data.empty else 'N/A'
    
    # Update the label text
    hover_geoid.text = [f"{geoid}"]
    hover_pop_value.text = [f"{value:,}"]



In [12]:
update_hover_fig('0101002')  # Example GEOID, replace with actual GEOID as needed
hover_fig

Figure(fig_margin={'top': 0.01, 'bottom': 0.01, 'left': 0.01, 'right': 0.01}, layout=Layout(height='110px', wi…

In [13]:
# adding widget on hover
widget_overview = WidgetControl(widget=hover_fig, position="bottomright")

m.add(widget_overview)

def on_hover(event, feature, **kwargs):
    global geoid
    geoid = feature["properties"]["GEOID"]
    update_hover_fig(geoid)  # Update the figure with the selected GEOID and demographic variable

geo.on_hover(on_hover)

m

Map(center=[np.float64(32.615677443261795), np.float64(-86.68073621090903)], controls=(ZoomControl(options=['p…

In [14]:
from ipywidgets import Button, Layout
import ipyleaflet
# I need a reset selectin button 
def reset_selection(button):
    selected_geoids.clear()  # Clear the selection
    update_highlight_layer([])  # Clear the highlight layer
    reset_female_percentage_graph(fig_percent)  # Reset
    update_right_panel(fig_percent)  # Update the right panel with the reset graph
    
reset_btn = Button(
    description='Reset Selection', 
    button_style='info',
    layout=Layout(width='auto', height='auto')
)

# changing button color
reset_btn.style.button_color = cc_blue

# adding reset button cabailities 
reset_btn.on_click(reset_selection)

widget_control = ipyleaflet.WidgetControl(widget=reset_btn, position='topright')
m.add_control(widget_control)

m

Map(center=[np.float64(32.615677443261795), np.float64(-86.68073621090903)], controls=(ZoomControl(options=['p…

In [15]:
from ipywidgets import HBox
import plotly.graph_objects as go

# Convert the static Figure into a live widget so it can sit in an HBox
right_panel = go.FigureWidget(fig_percent)
right_panel.layout.width = 350
right_panel.layout.height = 225

m_layout = HBox([m, right_panel])
m_layout

In [16]:
def update_right_panel_2(fig_percent):
    global m_layout
    
    # Convert the static Figure into a live widget so it can sit in an HBox
    right_panel = go.FigureWidget(fig_percent)
    right_panel.layout.width = 350
    right_panel.layout.height = 225

    m_layout = HBox([m, right_panel])
    
def update_right_panel(fig_percent):
    global right_panel
    right_panel.data = []
    for trace in fig_percent.data:
        right_panel.add_trace(trace)
    right_panel.layout.update(fig_percent.layout.to_plotly_json())

on click events
https://medium.com/swlh/how-to-use-mouse-events-on-ipyleaflet-4d002097efc0

In [17]:
def update_fig_side(geoids):
    # geoids can be a single value or a list — normalize to a list
    if not isinstance(geoids, (list, set, tuple)):
        geoids = [geoids]

    filtered_data = data[data['GEOID'].isin(geoids)]

    if filtered_data.empty:
        value = 'N/A'
    else:
        value = filtered_data[demographic_var].values  # array of values, one per selected region
        # decide here how to combine them if multiple selected, e.g.:
        # value = filtered_data[demographic_var].mean()  # average across selection
        # or keep as a list to show all selected values in the label/chart

    # Update the label text
    label_side.text = [str(value)]

In [18]:
def update_highlight_layer(geoids):
    # geoids can be a single value or a list — normalize to a list
    if not isinstance(geoids, (list, set, tuple)):
        geoids = [geoids]

    # Filter the GeoDataFrame for the selected GEOIDs
    filtered_gdf = gdf_small[gdf_small['GEOID'].astype(str).isin(geoids)]

    # Update the highlight layer with the new GeoJSON data
    highlight_layer.data = filtered_gdf.__geo_interface__

# Adding in gender decomp

In [19]:
print(data.head())

     GEOID  D001TPOP  D025MALE  D049FEMALE
0  0101001     55599     27908       27691
1  0101002     67916     32845       35071
2  0101003     34347     16594       17753
3  0101004     45525     22560       22965
4  0101005     36757     18254       18503


In [20]:
# now doing percentage of male and female population
data["D025MALE_PERCENT"] = round((data["D025MALE"] / data["D001TPOP"]) * 100, 2)
data["D049FEMALE_PERCENT"] = round((data["D049FEMALE"] / data["D001TPOP"]) * 100, 2)
print(data[["GEOID", "D025MALE_PERCENT", "D049FEMALE_PERCENT"]].head())

     GEOID  D025MALE_PERCENT  D049FEMALE_PERCENT
0  0101001             50.20               49.80
1  0101002             48.36               51.64
2  0101003             48.31               51.69
3  0101004             49.56               50.44
4  0101005             49.66               50.34


In [21]:
from graph_programs.female_percentage import update_female_percentage_graph

fig_percent_1 = update_female_percentage_graph(data, gdf_small, fig_percent)
fig_percent_1.show()

Now I want to incporate this into the graph

In [22]:
# initilaizing 
from graph_programs.female_percentage import update_female_percentage_graph

selected_geoids = set()

def handle_click(event=None, feature=None, **kwargs):
    global fig_percent
    geoid = feature["properties"]["GEOID"]

    if geoid in selected_geoids:
        selected_geoids.discard(geoid)
    else:
        selected_geoids.add(geoid)

    filtered = gdf_small[gdf_small['GEOID'].astype(str).isin(selected_geoids)]
    fig_percent = update_female_percentage_graph(
        demographics_data = data, 
        geoids_df = filtered, 
        fig_percent = fig_percent)
    update_right_panel(fig_percent)
    update_highlight_layer(list(selected_geoids))

geo.on_click(handle_click)

display(m_layout)


# Adding in Line Graph for Age Buckets